In [1]:
import os
os.chdir(os.path.join(os.getcwd(), '..'))
os.getcwd()

'd:\\learning\\learning\\programing\\portfolio_projects\\1_image'

In [2]:
from src.pipeline.mrisegmentator import MRISegmentator

In [3]:
segmentator = MRISegmentator()

base model checkpoint found loccaly
correction model checkpoint found loccaly
base model thresholds found loccaly
correction model thresholds found loccaly
Benchmarking optimal batch size...
Batch size 16 is valid
Segmentatator successfully initialized


In [4]:
from src.constants.local import DAGSHUB_API_REPO, DAGSHUB_BASE_THRESHOLDS

In [5]:
from pathlib import Path

In [ ]:
mri_sdf = segmentator(Path('data/extracted_data/train/case2/case2_day1/scans'), True, visualization=True)

torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])


interactive(children=(IntSlider(value=0, description='Slice', max=143), Dropdown(description='Class', options=…

In [30]:
import pandas as pd
df = pd.read_csv('data/preprocessed_data/final_data/test_post_model.csv')

In [31]:
df = df.drop(['prev_path','next_path','collapsed_mask_path','model_mask_path','mask_path', 'empty_slice'], axis = 1)

In [13]:
df

,collapsed_id,case,slice,path,mask_path
0,19,16,1,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...
1,19,16,2,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...
2,19,16,3,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...
3,19,16,4,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...
4,19,16,5,data\extracted_data\train\case16\case16_day0\s...,data\preprocessed_data\final_data\output_mask\...
...,...,...,...,...,...
1579,237,140,140,data\extracted_data\train\case140\case140_day1...,data\preprocessed_data\final_data\output_mask\...
1580,237,140,141,data\extracted_data\train\case140\case140_day1...,data\preprocessed_data\final_data\output_mask\...
1581,237,140,142,data\extracted_data\train\case140\case140_day1...,data\preprocessed_data\final_data\output_mask\...
1582,237,140,143,data\extracted_data\train\case140\case140_day1...,data\preprocessed_data\final_data\output_mask\...


In [39]:
df.model_mask[1]

WindowsPath('data/extracted_data/train/case16/case16_day0/scans_masks.npz')

In [37]:
df['dir_path'] = df.path.apply(lambda p: Path(p).parent)

In [17]:
for d in df.dir_path.unique():
    _ = segmentator(d, True)

torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 

In [34]:
df['model_mask'] = df.dir_path.apply(lambda p: p.parent/'scans_masks.npz')

In [24]:
import numpy as np

In [ ]:

best_f1 = 0.
test_len = len(df.model_mask.unique())
test_f1 = 0.
best_model_mask_path = None

for model_mask_p, mask_ps in df.groupby('model_mask')['out_transformed_mask_path']:
    model_mask = np.load(model_mask_p)['arr_0'].astype(bool)
    print(np.shape(np.load(mask_ps.to_numpy()[0])['arr_0']))
    mask = np.concat([[np.load(p)['arr_0']] for p in mask_ps.to_numpy()], axis = 0).astype(bool)

    tp = np.sum(mask & model_mask)
    f1 = (2*tp+1e-8)/(np.sum(mask)+np.sum(model_mask)+1e-8)

    if best_f1 < f1:
        best_f1 = f1
        best_model_mask_path = model_mask_p

    test_f1 +=f1

test_f1/=test_len


        
    


(3, 256, 256)
(3, 256, 256)
(3, 256, 256)
(3, 256, 256)
(3, 256, 256)
(3, 256, 256)
(3, 256, 256)
(3, 256, 256)
(3, 256, 256)
(3, 256, 256)
(3, 256, 256)


In [60]:
test_f1

np.float64(0.7293146005477028)

In [61]:
best_f1

np.float64(0.8194275293887844)

In [ ]:
mri_sdf = segmentator(best_model_mask_path.parent/'scans', True, visualization=True)

torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])
torch.Size([16, 256, 256]) torch.Size([16, 3, 256, 256]) torch.Size([16, 3, 256, 256])


interactive(children=(IntSlider(value=0, description='Slice', max=143), Dropdown(description='Class', options=…

In [98]:
test_masks_pathes = df[df['model_mask'] == best_model_mask_path[1]]['out_transformed_mask_path'].to_numpy()
test_masks = [np.load(p)['arr_0'].astype(bool) for p in test_masks_pathes]
model_mask = list(np.load(best_model_mask_path[0])['arr_0'].astype(bool))
mri_sdf.visualize_mask_3d(masks = model_mask, sigma = 0.5)

[Figure({
     'data': [{'color': 'red',
               'flatshading': False,
               'hovertemplate': 'Width: %{x:.1f}<br>Height: %{y:.1f}<br>Slice: %{z:.1f}<extra></extra>',
               'i': {'bdata': ('AgAEAAAABgAEAAgABgAIAAwADAAKAA' ... 't8e6V6fXt9e6R6e3useq96r3qxerF6'),
                     'dtype': 'i2'},
               'j': {'bdata': ('AQADAAMABQAFAAcABwAJAAsACgAAAA' ... 't4e317ent5e3p7rHp7e3t7fHt8e317'),
                     'dtype': 'i2'},
               'k': {'bdata': ('AAAAAAIABAADAAYABQAHAAoAAQABAA' ... 't9e6R6pHp6e5Z6onqvenx7sXp9e6V6'),
                     'dtype': 'i2'},
               'name': 'Large bowel',
               'opacity': 0.85,
               'type': 'mesh3d',
               'x': {'bdata': ('AAAaQ1KCGUMAABpDAAAbQwAAG0MAAB' ... 'CYQgAAmkIAAJxCAACWQgAAmEIAAJpC'),
                     'dtype': 'f4'},
               'y': {'bdata': ('AACYQgAAmEIuIpdCFACXQgAAmEIUAJ' ... 'ABQwAAAUMAAAFDAAACQwAAAkMAAAJD'),
                     'dtype': 'f4'},
              

In [91]:
best_model_mask_path

[WindowsPath('data/extracted_data/train/case121/case121_day0/scans_masks.npz'),
 WindowsPath('data/extracted_data/train/case121/case121_day0/scans_masks.npz'),
 WindowsPath('data/extracted_data/train/case121/case121_day16/scans_masks.npz')]

In [69]:
np.shape(test_masks)

(144, 3, 256, 256)

In [84]:
import numpy as np

# Initializing per-channel tracking variables
best_f1 = None
test_f1 = None
best_model_mask_path = None

test_len = len(df.model_mask.unique())


for model_mask_p, mask_ps in df.groupby('model_mask')['out_transformed_mask_path']:
    model_mask = np.load(model_mask_p)['arr_0'].astype(bool)
    mask = np.concat([[np.load(p)['arr_0']] for p in mask_ps.to_numpy()], axis = 0).astype(bool)
    print(mask.shape, model_mask.shape)
    print(np.shape(mask&model_mask))
    tp = np.sum(mask & model_mask, axis=(0, 2, 3))
    print(tp.shape)
    mask_sum = np.sum(mask, axis=(0, 2, 3))
    print(mask_sum.shape)
    model_mask_sum = np.sum(model_mask, axis=(0,2,3)) 
    print(model_mask_sum.shape)
    # Per-channel F1 score -> Shape: (C,)
    f1 = (2 * tp + 1e-8) / (mask_sum + model_mask_sum + 1e-8)

    # Initialize per-channel tracking arrays dynamically based on channel count C
    if best_f1 is None:
        num_channels = f1.shape[0]
        best_f1 = np.zeros(num_channels)
        test_f1 = np.zeros(num_channels)
        best_model_mask_path = [None] * num_channels

    # Update best F1 and path for each channel independently
    for c in range(num_channels):
        if best_f1[c] < f1[c]:
            best_f1[c] = f1[c]
            best_model_mask_path[c] = model_mask_p

    test_f1 += f1

test_f1 /= test_len

(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)
(144, 3, 256, 256) (144, 3, 256, 256)
(144, 3, 256, 256)
(3,)
(3,)
(3,)


In [85]:
best_f1

array([0.83990807, 0.76137066, 0.92532647])

In [86]:
test_f1

array([0.71056827, 0.60853925, 0.89176496])